<font face="楷体">

1. 导入必要的库
</font>

In [ ]:
# 1.导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示和图形美化参数
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

<font face="楷体">

2. 数据加载  
   加载数据集并查看前几行以了解特征结构。
</font>

In [ ]:
# 2.数据加载
# 从本地CSV文件读取数据
# diabetes.csv包含768个样本和9个特征列
data = pd.read_csv('diabetes.csv')

# 显示数据集的基本信息
print(f"数据集形状: {data.shape}")
print(f"总样本数: {data.shape[0]}")
print(f"特征数量: {data.shape[1] - 1}")  # 减去目标变量Outcome

print(f"\n前5行数据预览:")
print(data.head())

# 检查缺失值
print("\n缺失值统计:")
print(data.isnull().sum())

# 检查目标变量的分布
print("\n目标变量(Outcome)分布:")
print(data['Outcome'].value_counts())
print("\n目标变量比例:")
print(data['Outcome'].value_counts(normalize=True))

<font face="楷体">

3. 数据质量检查
</font>

In [ ]:
# 在医学数据中,某些特征值为0是不合理的(如血压、BMI等)
# 检查每个特征中0值的数量
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("\n不合理的零值统计(医学上不可能为0的特征):")
for col in zero_columns:
    zero_count = (data[col] == 0).sum()
    zero_percent = (zero_count / len(data)) * 100
    print(f"{col}: {zero_count} 个零值 ({zero_percent:.2f}%)")

<font face="楷体">

4. 数据预处理——处理零值
</font>

In [ ]:
# 创建数据副本以保留原始数据
data_processed = data.copy()

# 对于不合理的零值,使用该列的中位数进行填充
# 中位数比均值更稳健,不容易受到极端值的影响
print("以下特征中的零值均被替换为对应的中位数——")
for col in zero_columns:
    # 计算非零值的中位数
    median_value = data_processed[data_processed[col] != 0][col].median()
    # 将零值替换为中位数
    data_processed[col] = data_processed[col].replace(0, median_value)
    print(f"{col:<20} {median_value:>8.2f}")

<font face="楷体">

5. 数据可视化分析
</font>

In [ ]:
# 5.1特征分布直方图
print("\n5.1 所有特征的分布直方图——")
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

# 获取所有特征列
feature_columns = data_processed.columns[:]
# 为每个特征绘制直方图
for idx, col in enumerate(feature_columns):
    axes[idx].hist(data_processed[col], bins=30, color='skyblue', 
                   edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col} Distribution', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frequency', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 5.2特征相关性热力图
print("\n5.2 特征相关性矩阵热力图——")
plt.figure(figsize=(12, 10))
# 计算所有变量之间的皮尔逊相关系数
correlation_matrix = data_processed.corr()
# 绘制热力图
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

<font face="楷体">

6. 特征工程——数据分割
</font>

In [ ]:
# 6.1分离特征和目标变量
# X包含所有特征列,y包含目标变量(是否患糖尿病)
X = data_processed.drop('Outcome', axis=1)  # 删除Outcome列,剩余为特征
y = data_processed['Outcome']  # 提取Outcome列作为目标变量

print(f"特征矩阵形状: {X.shape}")
print(f"目标变量形状: {y.shape}")
print(f"\n特征列名:")
print(X.columns.tolist())

# 6.2划分训练集和测试集
# test_size=0.2表示20%的数据用于测试,80%用于训练
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

print(f"\n训练集大小: {X_train.shape[0]} 样本")
print(f"测试集大小: {X_test.shape[0]} 样本")
print(f"\n训练集中目标变量分布:")
print(y_train.value_counts())
print(f"\n测试集中目标变量分布:")
print(y_test.value_counts())

<font face="楷体">

7. 特征标准化
</font>

In [ ]:
# 创建标准化器对象
# StandardScaler会将每个特征转换为均值为0、标准差为1的分布
scaler = StandardScaler()

# 在训练集上拟合标准化器(计算均值和标准差)
# 然后对训练集进行转换
X_train_scaled = scaler.fit_transform(X_train)

# 使用训练集的均值和标准差对测试集进行转换
# 注意:测试集只用transform,不用fit,避免数据泄露
X_test_scaled = scaler.transform(X_test)

print(f"\n标准化前训练集特征范围示例 (第一个特征):")
print(f"最小值: {X_train.iloc[:, 0].min():.2f}, 最大值: {X_train.iloc[:, 0].max():.2f}")
print(f"\n标准化后训练集特征范围示例 (第一个特征):")
print(f"最小值: {X_train_scaled[:, 0].min():.2f}, 最大值: {X_train_scaled[:, 0].max():.2f}")
print(f"均值: {X_train_scaled[:, 0].mean():.2f}, 标准差: {X_train_scaled[:, 0].std():.2f}")

<font face="楷体">

8. 逻辑回归模型训练
</font>

In [ ]:
# 8.1创建逻辑回归模型
# max_iter=1000: 最大迭代次数,确保模型收敛
# solver='lbfgs': 优化算法,适合小数据集

# 创建逻辑回归模型
log_reg = LogisticRegression(max_iter=1000, solver='lbfgs')

# 8.2训练模型
# 使用标准化后的训练数据拟合模型
log_reg.fit(X_train_scaled, y_train)

# 8.3查看模型参数
print("\n模型参数:")
print(f"截距项 (intercept): {log_reg.intercept_[0]:.4f}")
print(f"\n各特征的系数 (coefficients):")
# 将特征名和对应的系数组合成DataFrame便于查看
coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0]
})
# 按系数绝对值排序,找出最重要的特征
coefficients_df['Abs_Coefficient'] = np.abs(coefficients_df['Coefficient'])
coefficients_df = coefficients_df.sort_values('Abs_Coefficient', ascending=False)
print(coefficients_df[['Feature', 'Coefficient']])

# 8.4可视化特征重要性(系数)
print("\n特征系数图——")
plt.figure(figsize=(10, 6))
colors = ['red' if x < 0 else 'green' for x in coefficients_df['Coefficient']]
plt.barh(coefficients_df['Feature'], coefficients_df['Coefficient'], 
         color=colors, alpha=0.7, edgecolor='black')
plt.xlabel('Coefficient Value', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Logistic Regression Feature Coefficients', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

<font face="楷体">

9. 模型预测
</font>

In [ ]:
# 9.1在训练集上进行预测
y_train_pred = log_reg.predict(X_train_scaled)
# 预测概率(返回每个样本属于各类别的概率)
y_train_pred_proba = log_reg.predict_proba(X_train_scaled)

# 9.2在测试集上进行预测
y_test_pred = log_reg.predict(X_test_scaled)
# 预测概率
y_test_pred_proba = log_reg.predict_proba(X_test_scaled)

<font face="楷体">

10. 模型性能评估
</font>

In [ ]:
# 10.1计算基本评估指标

# 训练集性能
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)
print("训练集性能:")
print(f"  准确率 (Accuracy):  {train_accuracy:.4f}")
print(f"  精确率 (Precision): {train_precision:.4f}")
print(f"  召回率 (Recall):    {train_recall:.4f}")
print(f"  F1分数 (F1-Score):  {train_f1:.4f}")

# 测试集性能
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
print("\n测试集性能:")
print(f"  准确率 (Accuracy):  {test_accuracy:.4f}")
print(f"  精确率 (Precision): {test_precision:.4f}")
print(f"  召回率 (Recall):    {test_recall:.4f}")
print(f"  F1分数 (F1-Score):  {test_f1:.4f}")

# 10.2 混淆矩阵
# 计算测试集的混淆矩阵
cm = confusion_matrix(y_test, y_test_pred)
print("\n混淆矩阵 (测试集):")
print(cm)

# 10.3详细分类报告
# classification_report提供每个类别的详细性能指标
print(classification_report(y_test, y_test_pred, 
                          target_names=['No Diabetes', 'Diabetes']))

# 10.4ROC曲线和AUC
# 计算ROC曲线的假阳性率和真阳性率
# 使用预测概率中属于正类(有糖尿病)的概率
fpr, tpr, thresholds = roc_curve(y_test, y_test_pred_proba[:, 1])
# 计算AUC (Area Under Curve) 值
roc_auc = roc_auc_score(y_test, y_test_pred_proba[:, 1])
print(f"AUC值: {roc_auc:.4f}")

# 绘制ROC曲线
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.4f})')
# 绘制对角线(随机猜测的基准线)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
         label='Random Guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', 
          fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<font face="楷体">

11. 预测概率分布分析
</font>

In [ ]:
# 分析模型预测概率的分布情况
# 获取测试集预测为正类(有糖尿病)的概率
test_proba_class1 = y_test_pred_proba[:, 1]

print(f"预测概率统计:")
print(f"  最小值: {test_proba_class1.min():.4f}")
print(f"  最大值: {test_proba_class1.max():.4f}")
print(f"  平均值: {test_proba_class1.mean():.4f}")
print(f"  中位数: {np.median(test_proba_class1):.4f}")

# 11.1绘制预测概率分布直方图
plt.figure(figsize=(12, 5))

# 子图1:所有样本的概率分布
plt.subplot(1, 2, 1)
plt.hist(test_proba_class1, bins=30, color='steelblue', 
         alpha=0.7, edgecolor='black')
plt.xlabel('Predicted Probability of Diabetes', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Distribution of Predicted Probabilities (All Samples)', 
          fontsize=12, fontweight='bold')
plt.axvline(x=0.5, color='red', linestyle='--', linewidth=2, 
            label='Decision Threshold (0.5)')
plt.legend()
plt.grid(axis='y', alpha=0.3)

# 子图2:按真实标签分组的概率分布
plt.subplot(1, 2, 2)
# 分别提取真实无糖尿病和有糖尿病样本的预测概率
proba_no_diabetes = test_proba_class1[y_test == 0]
proba_diabetes = test_proba_class1[y_test == 1]

plt.hist(proba_no_diabetes, bins=20, alpha=0.6, color='green', 
         label='True: No Diabetes', edgecolor='black')
plt.hist(proba_diabetes, bins=20, alpha=0.6, color='red', 
         label='True: Diabetes', edgecolor='black')
plt.xlabel('Predicted Probability of Diabetes', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Predicted Probabilities by True Label', 
          fontsize=12, fontweight='bold')
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=2, 
            label='Decision Threshold')
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

<font face="楷体">

12. 不同阈值下的性能分析
</font>

In [ ]:
# 逻辑回归默认使用0.5作为分类阈值
# 测试不同阈值
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = []

print("\n不同阈值下的性能对比:")
print("-" * 70)
print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 70)

for threshold in thresholds_to_test:
    # 根据当前阈值进行预测
    y_pred_threshold = (test_proba_class1 >= threshold).astype(int)
    
    # 计算各项指标
    acc = accuracy_score(y_test, y_pred_threshold)
    prec = precision_score(y_test, y_pred_threshold)
    rec = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)
    
    threshold_results.append({
        'Threshold': threshold,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print(f"{threshold:<12.1f} {acc:<12.4f} {prec:<12.4f} {rec:<12.4f} {f1:<12.4f}")

# 可视化不同阈值下的性能
threshold_df = pd.DataFrame(threshold_results)

plt.figure(figsize=(10, 6))
plt.plot(threshold_df['Threshold'], threshold_df['Accuracy'], 
         marker='o', label='Accuracy', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['Precision'], 
         marker='s', label='Precision', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['Recall'], 
         marker='^', label='Recall', linewidth=2)
plt.plot(threshold_df['Threshold'], threshold_df['F1-Score'], 
         marker='d', label='F1-Score', linewidth=2)
plt.xlabel('Classification Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Performance Metrics vs Classification Threshold', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<font face="楷体">

13. 错误分析
</font>

In [ ]:
# 找出预测错误的样本
# 假阳性:实际无糖尿病但预测为有糖尿病
false_positives = np.where((y_test == 0) & (y_test_pred == 1))[0]
# 假阴性:实际有糖尿病但预测为无糖尿病
false_negatives = np.where((y_test == 1) & (y_test_pred == 0))[0]

print(f"\n假阳性 (False Positives) 数量: {len(false_positives)}")
print(f"假阴性 (False Negatives) 数量: {len(false_negatives)}")
print(f"总错误数: {len(false_positives) + len(false_negatives)}")

# 分析假阳性样本的特征
if len(false_positives) > 0:
    print("\n假阳性样本特征统计 (前5个):")
    fp_indices = y_test.index[false_positives[:5]]
    fp_samples = data_processed.loc[fp_indices, X.columns]
    print(fp_samples)
    print(f"\n这些样本的预测概率:")
    for i, idx in enumerate(false_positives[:5]):
        print(f"  样本 {i+1}: {test_proba_class1[idx]:.4f}")

# 分析假阴性样本的特征
if len(false_negatives) > 0:
    print("\n假阴性样本特征统计 (前5个):")
    fn_indices = y_test.index[false_negatives[:5]]
    fn_samples = data_processed.loc[fn_indices, X.columns]
    print(fn_samples)
    print(f"\n这些样本的预测概率:")
    for i, idx in enumerate(false_negatives[:5]):
        print(f"  样本 {i+1}: {test_proba_class1[idx]:.4f}")